In [ ]:
# This is simply to verify the sequence in which the conditions/tasks were implemented. The order disovered is (stroop test --> baseline --> survery)

import pandas as pd
import os
from datetime import datetime, timezone

DATA_ROOT = os.path.join("..", "raw_data")
PILOT_ROOT = os.path.join(DATA_ROOT, "pilot")
SURVEY_ROOT = os.path.join(DATA_ROOT, "survey_gamification")

def get_times(path):
    """Return (start_unix, end_unix, start_str, end_str) from a bvp file."""
    if not os.path.exists(path):
        return None, None, None, None
    ts = pd.read_csv(path)["time"]
    fmt = lambda t: datetime.fromtimestamp(t, tz=timezone.utc).strftime("%H:%M:%S")
    return ts.iloc[0], ts.iloc[-1], fmt(ts.iloc[0]), fmt(ts.iloc[-1])

rows = []

# --- Pilot participants (IDs 0-10) ---
for pid in sorted(os.listdir(PILOT_ROOT), key=int):
    base = os.path.join(PILOT_ROOT, pid)
    conditions = {
        "cl":       os.path.join(base, "cognitive_load", "empatica_bvp.csv"),
        "baseline": os.path.join(base, "baseline",       "empatica_bvp.csv"),
    }
    # collect raw start times for ranking
    cond_data = {}
    for cond, path in conditions.items():
        s_unix, e_unix, s_str, e_str = get_times(path)
        cond_data[cond] = (s_unix, e_unix, s_str, e_str)

    # rank by start time (ascending)
    ranked = sorted(
        [(cond, d[0]) for cond, d in cond_data.items() if d[0] is not None],
        key=lambda x: x[1]
    )
    rank_map = {cond: i + 1 for i, (cond, _) in enumerate(ranked)}

    for cond, (s_unix, e_unix, s_str, e_str) in cond_data.items():
        rows.append({
            "participant":  int(pid),
            "dataset":      "pilot",
            "session":      "-",
            "condition":    cond,
            "start-time":   s_str,
            "end-time":     e_str,
            "sequence-rank": rank_map.get(cond),
        })

# --- Survey gamification participants (IDs 11-24, pre and post) ---
for pid in sorted(os.listdir(SURVEY_ROOT), key=int):
    base = os.path.join(SURVEY_ROOT, pid)
    for session in ["pre", "post"]:
        sess_dir = os.path.join(base, session)
        if not os.path.exists(sess_dir):
            continue
        conditions = {
            "cl":       os.path.join(sess_dir, "cognitive_load", "empatica_bvp.csv"),
            "baseline": os.path.join(sess_dir, "baseline",       "empatica_bvp.csv"),
            "survey":   os.path.join(sess_dir, "survey",         "empatica_bvp.csv"),
        }
        cond_data = {}
        for cond, path in conditions.items():
            s_unix, e_unix, s_str, e_str = get_times(path)
            cond_data[cond] = (s_unix, e_unix, s_str, e_str)

        ranked = sorted(
            [(cond, d[0]) for cond, d in cond_data.items() if d[0] is not None],
            key=lambda x: x[1]
        )
        rank_map = {cond: i + 1 for i, (cond, _) in enumerate(ranked)}

        for cond, (s_unix, e_unix, s_str, e_str) in cond_data.items():
            rows.append({
                "participant":   int(pid),
                "dataset":       "survey_gamification",
                "session":       session,
                "condition":     cond,
                "start-time":    s_str,
                "end-time":      e_str,
                "sequence-rank": rank_map.get(cond),
            })

df = pd.DataFrame(rows, columns=[
    "participant", "dataset", "session", "condition",
    "start-time", "end-time", "sequence-rank",
]).sort_values(["participant", "session", "sequence-rank"]).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
df

,participant,dataset,session,condition,start-time,end-time,sequence-rank
0,0,pilot,-,cl,08:46:07,08:51:06,1
1,0,pilot,-,baseline,08:51:48,08:55:11,2
2,1,pilot,-,cl,11:11:00,11:14:49,1
3,1,pilot,-,baseline,11:17:04,11:19:01,2
4,2,pilot,-,cl,06:21:12,06:25:49,1
5,2,pilot,-,baseline,06:28:24,06:31:25,2
6,3,pilot,-,cl,08:19:13,08:21:22,1
7,3,pilot,-,baseline,08:24:33,08:26:06,2
8,4,pilot,-,cl,06:57:40,07:02:04,1
9,4,pilot,-,baseline,07:03:32,07:06:35,2


../raw_data
